# 01_surrogate_baseline: Step 1 - surrogate baselines: ablation of the data steps, default model zoo,

```
and a replication of the Lyu et al. (2025) protocol.

A. Ablation (held-out test days, ExtraTrees & XGBoost defaults):
   raw 5-min -> + outlier smoothing -> + hourly aggregation -> + time-of-day/schedule
B. Default model zoo + weighted ensemble on the deployed (hourly) data
C. Paper protocol: office-1, first 12 weekdays, 5-min, smoothing, 7 features,
   random 70/15/15 row split (as AutoGluon-Tabular does by default)
Outputs -> outputs/surrogate/
```

Dikonversi dari `scripts/01_surrogate_baseline.py`. Jalankan dari folder `notebooks/`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent   # notebook runs from AI-OBEP-MLPaper/notebooks
sys.path.insert(0, str(ROOT / "src"))
from obep_mlpaper.data import (FEATURES, FEATURES_PAPER, TARGET, ROOM_AREA, add_time_features,
                               day_split, five_minute_dataset, hourly_dataset, load_room)
from obep_mlpaper.surrogate import WeightedEnsemble, metrics, model_zoo

out = ROOT / "outputs" / "surrogate"
out.mkdir(parents=True, exist_ok=True)

In [2]:
def fit_eval(name, make, Xtr, ytr, Xte, yte, op=None):
    m = make().fit(Xtr, ytr)
    p = np.clip(m.predict(Xte), 0, None)
    row = dict(model=name, **metrics(yte, p))
    if op is not None:
        row["R2_operating"] = metrics(yte[op], p[op])["R2"]
    return row, m, p

In [3]:
# ---------------------------------------------------------------- A. ablation
abl = []
steps = []
raw = add_time_features(load_room())
raw["y"] = raw["energy_kwh"] * 12 * 1000 / ROOM_AREA
steps.append(("1 raw 5-min, 7 paper features", raw, FEATURES_PAPER))
sm = five_minute_dataset(smooth=True)
sm["y"] = sm["target_kwh"] * 12 * 1000 / ROOM_AREA
steps.append(("2 + outlier smoothing (Hampel, adjacent mean)", sm, FEATURES_PAPER))
hr = hourly_dataset()
hr["y"] = hr[TARGET]
steps.append(("3 + hourly aggregation", hr, FEATURES_PAPER))
steps.append(("4 + time of day & HVAC schedule (deployed)", hr, FEATURES))
zoo = model_zoo()
for label, d, feats in steps:
    tr, te = day_split(d)
    X, y = d[feats].to_numpy(), d["y"].to_numpy()
    op = d["hvac_schedule"].to_numpy()[te] == 1
    for name in ("ExtraTrees", "XGBoost"):
        row, _, _ = fit_eval(name, lambda: model_zoo()[name], X[tr], y[tr], X[te], y[te], op)
        abl.append(dict(step=label, **row))
        print(f"{label:48s} {name:10s} R2={row['R2']:.3f}  R2_op={row['R2_operating']:.3f}  "
              f"CV(RMSE)={row['CVRMSE_pct']:.1f}%")
pd.DataFrame(abl).to_csv(out / "ablation_data_steps.csv", index=False)

1 raw 5-min, 7 paper features                    ExtraTrees R2=0.821  R2_op=0.084  CV(RMSE)=49.9%


1 raw 5-min, 7 paper features                    XGBoost    R2=0.781  R2_op=-0.060  CV(RMSE)=55.1%


2 + outlier smoothing (Hampel, adjacent mean)    ExtraTrees R2=0.847  R2_op=0.124  CV(RMSE)=45.5%


2 + outlier smoothing (Hampel, adjacent mean)    XGBoost    R2=0.840  R2_op=0.131  CV(RMSE)=46.4%


3 + hourly aggregation                           ExtraTrees R2=0.907  R2_op=0.307  CV(RMSE)=34.1%


3 + hourly aggregation                           XGBoost    R2=0.871  R2_op=0.201  CV(RMSE)=40.2%


4 + time of day & HVAC schedule (deployed)       ExtraTrees R2=0.939  R2_op=0.426  CV(RMSE)=27.6%


4 + time of day & HVAC schedule (deployed)       XGBoost    R2=0.944  R2_op=0.463  CV(RMSE)=26.5%


In [4]:
# ---------------------------------------------------------------- B. zoo on deployed data
d = hourly_dataset()
tr, te = day_split(d)
X, y, g = d[FEATURES].to_numpy(), d[TARGET].to_numpy(), d["day"].to_numpy()
op = d["hvac_schedule"].to_numpy()[te] == 1
rows = []
for name in zoo:
    row, _, _ = fit_eval(name, lambda: model_zoo()[name], X[tr], y[tr], X[te], y[te], op)
    rows.append(row)
makers = {k: (lambda k=k: model_zoo()[k]) for k in ("ExtraTrees", "RandomForest", "LightGBM", "XGBoost", "KNN", "MLP")}
ens = WeightedEnsemble(makers).fit(X[tr], y[tr], g[tr])
p = np.clip(ens.predict(X[te]), 0, None)
rows.append(dict(model="WeightedEnsemble", **metrics(y[te], p), R2_operating=metrics(y[te][op], p[op])["R2"]))
zoo_res = pd.DataFrame(rows)
zoo_res.to_csv(out / "baseline_zoo_test_metrics.csv", index=False)
print("\nDefault zoo (hourly, held-out days), ensemble weights:", ens.weight_dict())
print(zoo_res.round(3).to_string(index=False))


Default zoo (hourly, held-out days), ensemble weights: {'ExtraTrees': 0.372, 'RandomForest': 0.014, 'LightGBM': 0.103, 'XGBoost': 0.0, 'KNN': 0.0, 'MLP': 0.511}
           model    R2   RMSE    MAE  MAPE_pct  CVRMSE_pct  NMBE_pct  R2_operating
           Ridge 0.929 39.669 25.716    18.516      29.928    -2.981         0.343
             KNN 0.929 39.618 21.981    17.446      29.890    -2.496         0.327
    RandomForest 0.942 35.913 20.597    15.891      27.095    -3.177         0.447
      ExtraTrees 0.939 36.586 21.351    16.242      27.603    -2.838         0.426
        LightGBM 0.945 34.972 22.941    14.995      26.385    -3.038         0.502
         XGBoost 0.944 35.168 20.514    16.658      26.533    -4.193         0.463
             MLP 0.947 34.090 21.738    14.749      25.719    -1.868         0.531
WeightedEnsemble 0.948 33.802 21.007    14.976      25.502    -2.017         0.522


In [5]:
# ---------------------------------------------------------------- C. paper protocol
p12 = five_minute_dataset(n_days=12, smooth=True)
rng = np.random.RandomState(0)
idx = rng.permutation(len(p12))
n = len(idx)
itr, iva, ite = idx[:int(.7 * n)], idx[int(.7 * n):int(.85 * n)], idx[int(.85 * n):]
Xp, yp = p12[FEATURES_PAPER].to_numpy(), p12["target_kwh"].to_numpy()
prow, models = [], {}
for name in zoo:
    row, m, _ = fit_eval(name, lambda: model_zoo()[name], Xp[itr], yp[itr], Xp[ite], yp[ite])
    prow.append(row)
    models[name] = m
# AutoGluon-style L2 weighted ensemble: weights fit on the validation rows
from scipy.optimize import nnls
members = [k for k in models if k != "Ridge"]
Pv = np.column_stack([models[k].predict(Xp[iva]) for k in members])
w, _ = nnls(Pv, yp[iva])
w = w / w.sum()
pt = np.clip(np.column_stack([models[k].predict(Xp[ite]) for k in members]) @ w, 0, None)
prow.append(dict(model="WeightedEnsemble_L2", **metrics(yp[ite], pt)))
prow.append(dict(model="Lyu et al. 2025 (AutoGluon WeightedEnsemble_L2)", R2=0.933, MAPE_pct=26.094))
prow.append(dict(model="Lyu et al. 2025 (LSTM)", R2=0.921, MAPE_pct=19.358))
paper = pd.DataFrame(prow)
paper.to_csv(out / "paper_protocol_metrics.csv", index=False)
print("\nPaper protocol (12 days, 5-min, random rows):")
print(paper[["model", "R2", "RMSE", "MAPE_pct", "CVRMSE_pct"]].round(3).to_string(index=False))


Paper protocol (12 days, 5-min, random rows):
                                          model    R2  RMSE  MAPE_pct  CVRMSE_pct
                                          Ridge 0.682 0.802    33.443      65.485
                                            KNN 0.915 0.415    14.908      33.903
                                   RandomForest 0.942 0.342    11.570      27.974
                                     ExtraTrees 0.954 0.304    10.513      24.834
                                       LightGBM 0.944 0.337    11.789      27.562
                                        XGBoost 0.940 0.349    11.984      28.516
                                            MLP 0.928 0.381    13.059      31.088
                            WeightedEnsemble_L2 0.949 0.322    11.714      26.337
Lyu et al. 2025 (AutoGluon WeightedEnsemble_L2) 0.933   NaN    26.094         NaN
                         Lyu et al. 2025 (LSTM) 0.921   NaN    19.358         NaN
